In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/fake_job_postings.csv')
df.shape

(17880, 18)

In [2]:
before = len(df)
df.drop_duplicates(inplace=True)
after = len(df)

print(f"Removed: {before - after} duplicate rows")
print(f"Remaining: {after} rows")

Removed: 0 duplicate rows
Remaining: 17880 rows


In [3]:
# Check missing percentage per column
missing = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'missing_count'  : df.isnull().sum(),
    'missing_percent': missing
}).sort_values('missing_percent', ascending=False)

missing_df[missing_df['missing_count'] > 0]

,missing_count,missing_percent
salary_range,15012,83.96
department,11547,64.58
required_education,8105,45.33
benefits,7212,40.34
required_experience,7050,39.43
function,6455,36.10
industry,4903,27.42
employment_type,3471,19.41
company_profile,3308,18.50
requirements,2696,15.08


In [4]:
# Columns with >40% missing = not useful, drop them
cols_to_drop = missing_df[missing_df['missing_percent'] > 40].index.tolist()
df.drop(columns=cols_to_drop, inplace=True)

# Fill remaining text columns with placeholder
text_cols = ['company_profile', 'description', 'requirements', 'benefits']
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].fillna('Not Specified')

print(f"Dropped columns: {cols_to_drop}")
print(f"Remaining shape: {df.shape}")

Dropped columns: ['salary_range', 'department', 'required_education', 'benefits']
Remaining shape: (17880, 14)


In [7]:
# Combine only columns that still exist after dropping high-missing ones
text_cols_available = ['title', 'company_profile', 'description', 'requirements']

# Safely combine only available columns
df['full_text'] = df[text_cols_available].fillna('').apply(
    lambda row: ' '.join(row.values.astype(str)), axis=1
)

# Verify
df[['title', 'full_text', 'fraudulent']].head(3)

,title,full_text,fraudulent
0,Marketing Intern,"Marketing Intern We're Food52, and we've creat...",0
1,Customer Service - Cloud Video Production,Customer Service - Cloud Video Production 90 S...,0
2,Commissioning Machinery Assistant (CMA),Commissioning Machinery Assistant (CMA) Valor ...,0
